# Credit Card Fraud Detection - Environment Setup

This notebook initializes the Snowflake environment and loads synthetic credit card transaction data
for our Banking ML Demo. It covers:

1. Creating the database, schema, and warehouse
2. Generating synthetic transaction data with realistic fraud patterns
3. Loading data into Snowflake
4. Basic data exploration

**Snowflake ML Capabilities demonstrated in this series:**
- Feature Store (Feature Views, Entities, Dynamic Tables)
- Experiment Tracking
- Model Registry
- Model Serving (SPCS)
- Streamlit in Snowflake

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, count, avg, sum as sum_, min as min_, max as max_, stddev
from snowflake.snowpark.types import StructType, StructField, StringType, FloatType, IntegerType, TimestampType

session = get_active_session()
session.query_tag = {"origin": "ml_deep_dive", "name": "banking_fraud_detection", "notebook": "00_setup"}

# Configuration
DB = "BANKING_ML_DEMO"
SCHEMA = "FRAUD_DETECTION"
WH = "ML_DEMO_WH"

print(f"Session established: {session.get_current_role()}")

In [ ]:
# Create database, schema, and set context
session.sql(f"CREATE DATABASE IF NOT EXISTS {DB}").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}.{SCHEMA}").collect()
session.sql(f"CREATE WAREHOUSE IF NOT EXISTS {WH} WAREHOUSE_SIZE='X-SMALL' AUTO_SUSPEND=300 AUTO_RESUME=TRUE").collect()

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(WH)

print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")
print(f"Warehouse: {session.get_current_warehouse()}")

## Generate Synthetic Transaction Data

We generate ~100K credit card transactions with realistic patterns:
- **Normal transactions**: amounts following merchant-category distributions, business-hour timing
- **Fraudulent transactions (~2%)**: unusual amounts, late-night timing, geographic anomalies, higher online rate

In [ ]:
# Synthetic data generation parameters
N_TRANSACTIONS = 100_000
N_CUSTOMERS = 5_000
N_MERCHANTS = 500
FRAUD_RATE = 0.02
SEED = 42

rng = np.random.default_rng(SEED)

MERCHANT_CATEGORIES = ["grocery", "restaurant", "gas_station", "online_retail",
                       "electronics", "travel", "entertainment", "healthcare"]
CARD_TYPES = ["visa", "mastercard", "amex"]
CATEGORY_AMOUNTS = {
    "grocery": (75, 40), "restaurant": (45, 25), "gas_station": (55, 20),
    "online_retail": (120, 80), "electronics": (350, 200), "travel": (500, 300),
    "entertainment": (60, 35), "healthcare": (200, 150),
}
US_REGIONS = [(40.7, -74.0), (33.7, -84.4), (41.9, -87.6),
              (29.8, -95.4), (34.0, -118.2), (47.6, -122.3)]

start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 6, 30)
date_range_seconds = int((end_date - start_date).total_seconds())

# Generate customer and merchant profiles
home_regions = rng.choice(len(US_REGIONS), size=N_CUSTOMERS)
customers = pd.DataFrame({
    "CUSTOMER_ID": [f"CUST_{i:05d}" for i in range(N_CUSTOMERS)],
    "CUSTOMER_AGE": rng.integers(22, 75, size=N_CUSTOMERS),
    "CUSTOMER_INCOME": rng.integers(30000, 200000, size=N_CUSTOMERS),
    "HOME_LAT": [US_REGIONS[r][0] + rng.normal(0, 1) for r in home_regions],
    "HOME_LON": [US_REGIONS[r][1] + rng.normal(0, 1) for r in home_regions],
    "CARD_TYPE": rng.choice(CARD_TYPES, size=N_CUSTOMERS),
})

merchants = pd.DataFrame({
    "MERCHANT_ID": [f"MERCH_{i:04d}" for i in range(N_MERCHANTS)],
    "MERCHANT_CATEGORY": rng.choice(MERCHANT_CATEGORIES, size=N_MERCHANTS),
})

print(f"Customers: {len(customers)}, Merchants: {len(merchants)}")

In [ ]:
# Hour distribution for normal transactions (business-hour weighted)
normal_hour_weights = [1,1,1,1,1,2, 4,6,8,9,10,10, 10,9,8,8,9,10, 9,8,6,4,3,2]
normal_hour_probs = [w/sum(normal_hour_weights) for w in normal_hour_weights]

# Hour distribution for fraud (late-night weighted)
fraud_hour_weights = [8,9,10,10,9,7, 4,3,3,3,4,4, 4,4,3,3,3,4, 5,5,6,7,8,8]
fraud_hour_probs = [w/sum(fraud_hour_weights) for w in fraud_hour_weights]

n_fraud = int(N_TRANSACTIONS * FRAUD_RATE)
n_normal = N_TRANSACTIONS - n_fraud

def make_transactions(n, is_fraud=False):
    cust_idx = rng.choice(len(customers), size=n)
    merch_idx = rng.choice(len(merchants), size=n)
    cust_rows = customers.iloc[cust_idx].reset_index(drop=True)
    merch_rows = merchants.iloc[merch_idx].reset_index(drop=True)
    
    amounts = []
    for cat in merch_rows["MERCHANT_CATEGORY"]:
        mean, std = CATEGORY_AMOUNTS[cat]
        if is_fraud:
            amt = max(50.0, rng.normal(mean * rng.uniform(2.5, 8.0), std * 2))
        else:
            amt = max(1.0, rng.normal(mean, std))
        amounts.append(round(amt, 2))
    
    hour_probs = fraud_hour_probs if is_fraud else normal_hour_probs
    hours = rng.choice(24, size=n, p=hour_probs)
    random_secs = rng.integers(0, date_range_seconds, size=n)
    timestamps = [
        (start_date + timedelta(seconds=int(s))).replace(hour=int(h), minute=int(rng.integers(0, 60)))
        for s, h in zip(random_secs, hours)
    ]
    
    loc_noise = 5 if is_fraud else 0.3
    countries = rng.choice(["US","US","US","US","US","US","US","US","UK","CA","MX","BR","DE","FR","NG","CN"], size=n) if is_fraud else ["US"]*n
    online_probs = [0.3, 0.7] if is_fraud else [0.6, 0.4]
    
    return pd.DataFrame({
        "CUSTOMER_ID": cust_rows["CUSTOMER_ID"].values,
        "MERCHANT_ID": merch_rows["MERCHANT_ID"].values,
        "AMOUNT": amounts,
        "TIMESTAMP": timestamps,
        "MERCHANT_CATEGORY": merch_rows["MERCHANT_CATEGORY"].values,
        "LOCATION_LAT": cust_rows["HOME_LAT"].values + rng.normal(0, loc_noise, n),
        "LOCATION_LONG": cust_rows["HOME_LON"].values + rng.normal(0, loc_noise, n),
        "CARD_TYPE": cust_rows["CARD_TYPE"].values,
        "IS_ONLINE": rng.choice([0, 1], size=n, p=online_probs),
        "COUNTRY": countries,
        "CUSTOMER_AGE": cust_rows["CUSTOMER_AGE"].values,
        "CUSTOMER_INCOME": cust_rows["CUSTOMER_INCOME"].values,
        "IS_FRAUD": int(is_fraud),
    })

normal_txns = make_transactions(n_normal, is_fraud=False)
fraud_txns = make_transactions(n_fraud, is_fraud=True)

df = pd.concat([normal_txns, fraud_txns], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
df["TRANSACTION_ID"] = [f"TXN_{i:06d}" for i in range(len(df))]

cols = ["TRANSACTION_ID", "CUSTOMER_ID", "MERCHANT_ID", "AMOUNT", "TIMESTAMP",
        "MERCHANT_CATEGORY", "LOCATION_LAT", "LOCATION_LONG", "CARD_TYPE",
        "IS_ONLINE", "COUNTRY", "CUSTOMER_AGE", "CUSTOMER_INCOME", "IS_FRAUD"]
df = df[cols].sort_values("TIMESTAMP").reset_index(drop=True)

print(f"Generated {len(df):,} transactions")
print(f"Fraud rate: {df['IS_FRAUD'].mean():.2%}")
print(f"Date range: {df['TIMESTAMP'].min()} to {df['TIMESTAMP'].max()}")

In [ ]:
# Load data into Snowflake
session.sql("CREATE OR REPLACE TABLE RAW_TRANSACTIONS ("
    "TRANSACTION_ID VARCHAR, CUSTOMER_ID VARCHAR, MERCHANT_ID VARCHAR, "
    "AMOUNT FLOAT, TIMESTAMP TIMESTAMP_NTZ, MERCHANT_CATEGORY VARCHAR, "
    "LOCATION_LAT FLOAT, LOCATION_LONG FLOAT, CARD_TYPE VARCHAR, "
    "IS_ONLINE INTEGER, COUNTRY VARCHAR, CUSTOMER_AGE INTEGER, "
    "CUSTOMER_INCOME INTEGER, IS_FRAUD INTEGER)").collect()

snowpark_df = session.write_pandas(df, table_name="RAW_TRANSACTIONS", auto_create_table=False, overwrite=True)

raw_txns = session.table("RAW_TRANSACTIONS")
print(f"Loaded {raw_txns.count():,} rows into RAW_TRANSACTIONS")
raw_txns.show(10)

## Data Exploration

In [ ]:
# Basic exploration
raw_txns = session.table("RAW_TRANSACTIONS")

print("=== Schema ===")
for field in raw_txns.schema.fields:
    print(f"  {field.name}: {field.datatype}")

print(f"\n=== Summary ===")
print(f"Total transactions: {raw_txns.count():,}")
print(f"Unique customers: {raw_txns.select('CUSTOMER_ID').distinct().count():,}")
print(f"Unique merchants: {raw_txns.select('MERCHANT_ID').distinct().count():,}")

In [ ]:
# Fraud distribution
from snowflake.snowpark.functions import when, lit, round as round_

fraud_dist = raw_txns.group_by("IS_FRAUD").agg(
    count("*").alias("COUNT"),
    avg("AMOUNT").alias("AVG_AMOUNT"),
    min_("AMOUNT").alias("MIN_AMOUNT"),
    max_("AMOUNT").alias("MAX_AMOUNT"),
    stddev("AMOUNT").alias("STDDEV_AMOUNT")
)
print("=== Fraud Distribution ===")
fraud_dist.show()

# Transactions by merchant category
print("\n=== Transactions by Merchant Category ===")
raw_txns.group_by("MERCHANT_CATEGORY").agg(
    count("*").alias("COUNT"),
    avg("AMOUNT").alias("AVG_AMOUNT"),
    avg("IS_FRAUD").alias("FRAUD_RATE")
).sort("MERCHANT_CATEGORY").show()

# Card type distribution
print("\n=== Transactions by Card Type ===")
raw_txns.group_by("CARD_TYPE").agg(
    count("*").alias("COUNT"),
    avg("IS_FRAUD").alias("FRAUD_RATE")
).show()

## Next Steps

Data is loaded and ready. Proceed to **`01_feature_engineering.ipynb`** to:
- Initialize the Snowflake Feature Store
- Create Entities and Feature Views
- Generate training datasets from registered features